# (1) import libraries

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# (2) data preparation

In [ ]:
from scipy.stats import multivariate_normal
A = 32j
x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x_train = np.column_stack([x1.flat, x2.flat]).astype('float32')
y_train = multivariate_normal.pdf(x_train, mean=[0, 0], cov=np.eye(2)).astype('float32')
A = 10j
x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x_test = np.column_stack([x1.flat, x2.flat]).astype('float32')
y_test = multivariate_normal.pdf(x_test, mean=[0, 0], cov=np.eye(2)).astype('float32')

# (3) MLP

In [ ]:
class GaussMLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, hidden), nn.Sigmoid(), nn.Linear(hidden, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x).squeeze(1)

def train_gauss(hidden=16, samples=800, epochs=100, lr=0.01):
    model = GaussMLP(hidden)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    xb = torch.from_numpy(x_train[:samples]); yb = torch.from_numpy(y_train[:samples])
    for _ in range(epochs):
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    with torch.no_grad():
        train_error = criterion(model(torch.from_numpy(x_train)), torch.from_numpy(y_train)).item()
        test_error = criterion(model(torch.from_numpy(x_test)), torch.from_numpy(y_test)).item()
    return train_error, test_error

# (4) training and evaluating MLP

### (4.1) Number of training samples (A) (no. of hidden nodes (B) = 16 and no. of epochs (C) = 100)

In [ ]:
train_error = []; test_error = []
for samples in range(100, 1001, 100):
    train_loss, test_loss = train_gauss(samples=samples)
    train_error.append(train_loss); test_error.append(test_loss)
print('train errors:', train_error)
print('test errors:', test_error)

### (4.2) Number of hidden nodes (B) (no. of training samples (A) = 800 and no. of epochs (C) = 100)

In [ ]:
train_error = []; test_error = []
for hidden in [8, 16, 32, 64, 128, 256, 512]:
    train_loss, test_loss = train_gauss(hidden=hidden)
    train_error.append(train_loss); test_error.append(test_loss)
print('hidden-width experiment:', list(zip([8, 16, 32, 64, 128, 256, 512], test_error)))

### (4.3) Number of epochss (C) (no. of training samples (A) = 800 and no. of hidden nodes (B) = 128)